In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Random Sampling can be done using train_test_split function present in the sklearn library.

In [ ]:
from sklearn.model_selection import train_test_split
def random_split(dataset,t_size):
    train_set,test_set=train_test_split(dataset,test_size=t_size,random_state=7)
    return train_set,test_set

Random Sampling can be safely used only if the data is large enough or else we might run in sampling bias. Instead using a Stratified Sampling is advisable which divides population into homogenous subgroups called strata. Also keep in mind that large number of stratum could also produce bias ensure that each stratum contains a significant amount of instances. Having multiple splits can help in evaluating the model's performance.

In [ ]:

from sklearn.model_selection import StratifiedShuffleSplit

def stratified_split(dataset,target_attribute,n,t_size):
    splitter =StratifiedShuffleSplit(n_splits=n, test_size=t_size, random_state=7)
    strats_split=[]
    for train_index,test_index in splitter.split(dataset,dataset[target_attribute]):
        strats_train=dataset.iloc[train_index]
        strats_test=dataset.iloc[test_index]
        strats_split.append([strats_train,strats_test])
    return strats_split
    #strat_train,strat_test=train_test_split(dataset,test_size=t_size,random_state=7,stratified=dataset[target_attribute]) #This can be used to produce a single split

Use Correlation for different features in the dataset

In [ ]:
def correlation(dataset,target_attribute):
    corr_matrix=dataset.corr(numeric_only=True)
    return corr_matrix[target_attribute].sort_values(ascending=False)

Also note that datasets may contain missing values and there are 3 ways to deal with this.
1. Drop the rows with missing values.
2. Drop the attribute completely.
3. Replace the missing value (zero,mean,median,etc.). This is called imputation.

Imputation can be done using fillna(). Sklearn also has a SimpleImputer class for this.

In [ ]:
from sklearn.impute import SimpleImputer
def imputation(dataset,strategy_):
    imputer=SimpleImputer(strategy=strategy_) # Strategy includes mean, median, most_frequent, constant if constant is used then add fill_value.
    dataset_req=dataset.select_dtypes([np.number])
    imputer.fit(dataset_req)
    # The results will be stored in the statistics_ instance variable.
    return imputer.transform(dataset_req)

sklearn.impute also contains KNNImputer, IterativeImputer.

All objects share a consistent and simple interface:

Estimators:

Any object that can estimate some parameters based on a dataset is called an estimator (e.g., a SimpleImputer is an estimator). The estimation itself is performed by the fit() method, and it takes a dataset as a parameter, or two for supervised learning algorithms—the second dataset contains the labels. Any other parameter needed to guide the estimation process is considered a hyperparameter (such as a SimpleImputer’s strategy), and it must be set as an instance variable (generally via a constructor parameter).

Transformers:

Some estimators (such as a SimpleImputer) can also transform a dataset; these are called transformers. Once again, the API is simple: the transformation is performed by the transform() method with the dataset to transform as a parameter. It returns the transformed dataset. This transformation generally relies on the learned parameters, as is the case for a SimpleImputer. All transformers also have a convenience method called fit_transform(), which is equivalent to calling fit() and then transform() (but sometimes fit_transform() is optimized and runs much faster).

Predictors:

Finally, some estimators, given a dataset, are capable of making predictions; they are called predictors. For example, the LinearRegression model in the previous chapter was a predictor: given a country’s GDP per capita, it predicted life satisfaction. A predictor has a predict() method that takes a dataset of new instances and returns a dataset of corresponding predictions. It also has a score() method that measures the quality of the predictions, given a test set (and the corresponding labels, in the case of supervised learning algorithms).

Inspection:

All the estimator’s hyperparameters are accessible directly via public instance variables (e.g., imputer.strategy), and all the estimator’s learned parameters are accessible via public instance variables with an underscore suffix (e.g., imputer.statistics_).




Note that transform() outputs an array.

Handling Text and Categorical Attributes.

ML algos prefer to work in numbers. Hence changing text to numbers is important and can be done using OrdinalEncoder.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
def ordinal_encoder(dataset,target_attribute):
    encoder=OrdinalEncoder()
    return encoder.fit_transform(dataset[target_attribute]) # List of categories can be accessed using categories_ instance variable

One problem with this is that ML algo will assume that nearby values are more similar that to distant which might be true in some cases like good, better, best.
To overcome this we can create a vector for each instance and 1 will be assigned to the corrosponding category and the rest zero. For example for an attribute with three categories an instance belonging to the 2nd class will be represented as [0 1 0]. This can be implemented using OneHotEncoder() class.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
def one_hot_encoder(dataset,target_attribute):
    encoder=OneHotEncoder()
    return encoder.fit_transform(dataset[target_attribute]) # It returns a sparse matrix you can get an array by passing False to the sparse_output argument in OneHotEncoder.

Feature Scaling and Transformation
It's always best to scale the features or else the ML algo might will be biased towards a particular feature. Always use fit(), fit_transform() only to the training set. You can then use the fitted trained scaler to transform test, validation or new data set.

MinMaxScaler transformer shifts the values and rescale them to be in the range 0 to 1. The scale can be changed using the feature_range hyperparameter. It is sensitive to outliers.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
def min_max_scaler(dataset,lowerbound,upperbound):
    min_max=MinMaxScaler(feature_range=(lowerbound,upperbound))
    return min_max.fit_transform(dataset)

StandardScaler transformer produces a zero mean dataset. It is less affected by outliers.

In [ ]:
from sklearn.preprocessing import StandardScaler
def standard_scaler(dataset):
    scaler=StandardScaler() # Set hyperparameter with_mean to False for OneHotEncoded Features.
    return scaler.fit_transform(dataset)

When a feature distribution has a heavy tail(values far from the mean are not exponentially rare) the scaler squashes most values to a smaller range which is not advisable. So before scaling the feature transforming the feature to a lower tail is important. This can be done by taking square root (or raising it to a power between 0 and 1). If it has a very long tail(power law distribution) then taking its logarithm might work. You should always try to make the distribution Gaussian.
Another way is to bucketizing the features. This means making equal-sized buckets and replacing feature values with its index of the bucket. Here we need don't need to scale it just divide the number of buckets.

When a feature has a multimodal distribution (with one or more clear peaks) use OneHotEncoder.
Another approach is to add a new feature for each of the mode representing the similarity between the feature and that particular mode. This similarity is calculated using radial basis function(rbf) any function that only depends on the distance between input value and a fixed point. Commonly used is Gaussian RBF with a hyperparameter gamma it determines how quickly the similarity measure decays.

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel
def feature_similarity(dataset,target_attribute,mode,g):
    return rbf_kernel(dataset[[target_attribute]],[[mode]],gamma=0.1)

As we are scaling the training set, the target value will also be scaled which can be reverted again using inverse_transform(). For regression using TransformedTargetRegressor from sklearn.compose is advisable

Custom Transformers:
 You can use FunctionTransformer() if you need a custom no training transform such as log, square root, etc.

In [ ]:
from sklearn.preprocessing import FunctionTransformer
log_transformer=FunctionTransformer(np.log, inverse_func=np.exp)

Transformer Pipelines: As there are many transformation steps scikit learn has a Pipeline class for this purpose.

In [ ]:
from sklearn.pipeline import Pipeline
num_pipeline=Pipeline([('imputer',SimpleImputer()),'standardize',StandardScaler()]) #Here everything should be a estimator expect the last one which could be a transformer, predictor or a estimator.
from sklearn.pipeline import make_pipeline

num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()) # without a name for the estimators

Pipelines support indexing. steps attribute can be used to access the estimators.
ColumnTransformer can be used to transform both the numerical and categorical columns.

In [ ]:
cat_pipeline=make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore'))

from sklearn.compose import ColumnTransformer
def preprocessing(num_attributes,cat_attributes):
    return ColumnTransformer([('num',num_pipeline,num_attributes),('cat',cat_pipeline,cat_attributes)])

# Since listing every columns is not convenient we used this instead
from sklearn.compose import make_column_selector, make_column_transformer

preprocessing = make_column_transformer((num_pipeline, make_column_selector(dtype_include=np.number)),(cat_pipeline, make_column_selector(dtype_include=object)))
#You can get the column names using preprocessing.get_feature_names_out()

Model Evaluation: Usually training with the training set might result in overfitting. Another alternative is to use k fold cross-validation feature. Here the training set is split into k nonoverlapping subsets called folds.

In [ ]:
from sklearn.model_selection import cross_val_score
def cross_score(model,dataset,labels,k):
    return -cross_val_score(model,dataset,labels,scoring="neg_root_mean_squared_error",cv=k) # It expects a utility function not a cost function

Fine-Tuning the model:
Fine-tuning the hyperparameters manually might be difficult instead using GridSearchCV class can be used for this purpose. You need to input the hyperparameters you want to tune and it will use cross-validation to evaluate all possible combinations. But RandomizedSearchCV is often preferable.
Several Benefits of using RandomizedSearchCV are:
If some of your hyperparameters are continuous (or discrete but with many possible values), and you let randomized search run for, say, 1,000 iterations, then it will explore 1,000 different values for each of these hyperparameters, whereas grid search would only explore the few values you listed for each one.

Suppose a hyperparameter does not actually make much difference, but you don’t know it yet. If it has 10 possible values and you add it to your grid search, then training will take 10 times longer. But if you add it to a random search, it will not make any difference.

If there are 6 hyperparameters to explore, each with 10 possible values, then grid search offers no other choice than training the model a million times, whereas random search can always run for any number of iterations you choose.